# Accessing images from the loc.gov JSON API for image analysis

The digital collections' images available from the [Library of Congress website](https://www.loc.gov) are an amazing resource for study and analysis using digital methods. This notebook shows how you can use the loc.gov JSON API and Python to access sets of images.

More information about API is at [About the loc.gov JSON API](https://libraryofcongress.github.io/data-exploration/). For more example code in Jupyter notebooks, check out [LC for Robots](https://labs.loc.gov/lc-for-robots). 

This tutorial contains several stages:

1. [Context](#Context)
2. [Identifying items](#Identifying-items)
3. [Calling the API and retrieving image URLs](#Calling-the-API-and-retrieving-image-URLs)
4. [Accessing images](#Accessing-images)
5. [Conclusion](#Conclusion)
   
## Version
---
Version: 2

Last Run: July 14, 2025 (Python 3.12)


---
Author Information:


Written by Laura Wrubel, Visiting Scholar 2018

Edited by Sabrina Templeton, Junior Fellow 2025

---

## Prerequisites
None. However, note that running this notebook will download files onto your machine. 
 

## Context

### Rights and access
Rights and restrictions, including copyright, affect how you can use images, particularly if you want to publish, display, or otherwise distribute them. You can read more in [About Copyright and the Collections](https://www.loc.gov/legal/). This notebook uses images from the Prints & Photographs Division (P&P), and you can read more about restrictions guidance for these materials at [Copyright and Other Restrictions That Apply to Publication/Distribution of Images:
Assessing the Risk of Using a P&P Image](http://www.loc.gov/rr/print/195_copr.html). There is also information about rights specific to the collection and item in several fields in the API response. See the end of this notebook for relevant fields.

### Image sizes and information about collections
In many cases, the images from the Prints & Photographs Division available via the API are a small thumbnail (150px on one side). Consider whether this is sufficient for your research methods and needs. If you need to work with higher resolution images than are available via the API, you'll need to consult with staff who work with the collection(s) to determine whether those are available for use on-site at the Library of Congress and what restrictions apply. Here are some examples of images that are 150px on one side: 

![https://www.loc.gov/item/2017691815/](https://cdn.loc.gov/service/pnp/fsa/8b02000/8b02800/8b02895_150px.jpg)
![https://www.loc.gov/item/2011645392/](https://cdn.loc.gov/service/pnp/ppmsca/31200/31267_150px.jpg)

It can also be helpful to discuss with staff your intended analysis, as factors such as the sources of metadata and how items were digitized may affect your findings. The [Ask a Librarian](http://www.loc.gov/rr/askalib/) service can help you get in touch with staff who work with the collections.

### More APIs for working with images
Outside of the Prints & Photographs Division images, most of the Library of Congress images are available from an [IIIF API](https://www.loc.gov/apis/micro-services/image-services/), including for example those of newspapers in the [Chronicling America](https://chroniclingamerica.loc.gov/) collection, [images from the Mansucript Division](https://www.loc.gov/search/?fa=online-format:image%7Cpartof:manuscript+division), and [images of maps](https://www.loc.gov/maps/?fa=online-format:image). The IIIF API allows you to zoom, rotate, resize, and work with images in more complex ways. For a brief intro to using the IIIF API, see [this Jupyter notebook on IIIF](https://github.com/LibraryOfCongress/data-exploration/blob/master/loc.gov%20IIIF%20API/IIIF.ipynb). 
 

## Identifying items 

So let's get started accessing images! If you haven't already, browse or search the [Library of Congress Digital Collections](https://www.loc.gov/collections) website to see what is available and refine your search parameters. The website lets you search over 500 curated collections by keyword, format, topic, and division. While the API documentation has [detailed information on parameters](https://www.loc.gov/apis/json-and-yaml/) you can use in your search, searching the collections often gives you good context that is useful in understanding the context of an image within its collection.

Once you've found a search that targets the items that interest you, **copy the URL**. That will be the **base URL** for your API request. 

For example: 

``https://www.loc.gov/collections/baseball-cards/``

``https://www.loc.gov/photos/?q=bridges&dates=1800/1899``

### Calling the API and retrieving image URLs

The function below will call the API, adding the following parameters:
* ``fo=json`` to get JSON format in response
* ``c=100`` a count of 100 results in each response, rather than the default 25
* ``at=results,pagination`` provide only the ``results`` and ``pagination`` parts of the response. The API response is otherwise very long with information
we don't need.


Each of the results in the `results` field will have a field called ``image_url``: 

``
"image_url": [
"//cdn.loc.gov/service/pnp/cph/3f00000/3f05000/3f05300/3f05332_150px.jpg",
"//cdn.loc.gov/service/pnp/cph/3f00000/3f05000/3f05300/3f05332_150px.jpg#h=150&w=100",
"//cdn.loc.gov/service/pnp/cph/3f00000/3f05000/3f05300/3f05332t.gif#h=150&w=100",
"//cdn.loc.gov/service/pnp/cph/3f00000/3f05000/3f05300/3f05332r.jpg#h=640&w=425",
"//cdn.loc.gov/service/pnp/cph/3f00000/3f05000/3f05300/3f05332v.jpg#h=1024&w=680"
],
``

For images from the Prints & Photographs Division, the last one listed is usually the largest of the image files publicly available. Below, the function retrieves the last item in the ``image_url`` array. 

In [12]:
import requests

def get_image_urls(url, items=[]):
    '''
    Retrieves the image URLs for items that have public URLs available. 
    Skips over items that are for the colletion as a whole or web pages about the collection.
    Handles pagination. 

    Args: 
        url (str): The URL to request a collection.
        items (list, optional): The list that fetched item URLs will get added to.

    Returns:
        list: The item URLS from the collection.  
    '''
    # request pages of 100 results at a time
    jsonParams = {"fo": "json", "c": 100, "at": "results,pagination"} 
    call = requests.get(url, params=jsonParams)
    data = call.json() 
    results = data['results']
    
    for result in results:
        # don't try to get images from the collection-level result
        if "collection" not in result.get("original_format") and "web page" not in result.get("original_format"):
            # take the last URL listed in the image_url array
            if result.get("image_url"):
                item = result.get("image_url")[-1]
                items.append(item)
    if data["pagination"]["next"] is not None: # make sure we haven't hit the end of the pages
        next_url = data["pagination"]["next"]
        print(f"getting next page: {next_url}")
        time.sleep(3) # timer to avoid API rate limits
        get_image_urls(next_url, items) 
        
    return items


The Library of Congress has a digitized collection of Baseball Cards collection from the late 19th and eary 20th century. Here's an example:

!["https://www.loc.gov/collections/static/baseball-cards/images/Ward0006.jpg"](https://www.loc.gov/collections/static/baseball-cards/images/Ward0006.jpg)

Let's retrieve URLs for images in the Baseball Cards collection. The base URL below will filter the results to those from the year 1888, allowing us to work with a more manageable number of files for the purpose of this notebook:

In [13]:
image_urls = get_image_urls("https://www.loc.gov/collections/baseball-cards/?dates=1888") # remove the date parameter here to download the entire collection

How many URLs were retrieved?

In [14]:
len(image_urls)

58

As mentioned above, we filtered by date to receive a more manageable number of images for the purposes of this tutorial, as the total number of images in the Baseball Cards collection is over 2000. You can easily remove the date parameter above if you want to download the entirety of the collection.

Here are the URLs retrieved from the ``image_url`` field in the first five results in the API response. As you can see by the way the files have been named, these have one dimension being 1024 pixels. 

In [15]:
image_urls[:5]

['https://tile.loc.gov/storage-services/service/pnp/bbc/0300/0380/0382fv.jpg#h=1024&w=572',
 'https://tile.loc.gov/storage-services/service/pnp/bbc/0500/0530/0536fv.jpg#h=1024&w=561',
 'https://tile.loc.gov/storage-services/service/pnp/bbc/0500/0530/0537fv.jpg#h=1024&w=569',
 'https://tile.loc.gov/storage-services/service/pnp/bbc/0500/0550/0550fv.jpg#h=1024&w=554',
 'https://tile.loc.gov/storage-services/service/pnp/bbc/0500/0550/0551fv.jpg#h=1024&w=554']

You could pass this list of URLs to a tool like wget to collect the images. A helpful tutorial is [Automated Downloading with wget](https://programminghistorian.org/lessons/automated-downloading-with-wget). 

In [5]:
import json

# request JSON for a single item
r = requests.get("https://www.loc.gov/item/2007678540", params={"fo": "json"})
r_data = r.json()

In [15]:
import requests
import json

r = requests.get("https://www.loc.gov/item/2007678540/?fo=json")

# Check if the response is empty
if not r.text:
    print("The response is empty.")
else:
    try:
        r_data = r.json()
        print(r_data)
    except ValueError as e:
        print(f"Failed to decode JSON: {e}")
        print(f"Response content: {r.text[:1000]}")  # Print the first 1000 characters of the response

Failed to decode JSON: Expecting value: line 1 column 1 (char 0)
Response content: <!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,"Helvetica Neue",Arial,"Noto Sans",sans-serif,"Apple Color Emoji","Segoe UI Emoji","Segoe UI Symbol","Noto Color Emoji"}body{display:flex;flex-direction:column;height:100vh;min-height:100vh}.main-content{margin:8rem auto;padding-left:1.5rem;max-width:60rem}@media (width <= 720px){.main-content{margin-top:4rem}}.h2{line-height:2.25rem;font-size:1.5rem;font-weight:500}@media (width <= 720px){.h2{line-height:1.5rem;font-si

**rights_information**

In [7]:
print(json.dumps(r_data["item"]["rights_information"], indent=2))

"No known restrictions on publication."


**rights_advisory**

In [9]:
print(json.dumps(r_data["item"]["rights_advisory"], indent=2))

NameError: name 'r_data' is not defined

**rights**

In [10]:
print(json.dumps(r_data["item"]["rights"], indent=2))

NameError: name 'r_data' is not defined

**access_restricted**

If access is restricted, this field will be true. 

In [11]:
print(json.dumps(r_data["item"]["access_restricted"], indent=2))

NameError: name 'r_data' is not defined

## Accessing images

Now that you have the URLs for images, you can access them as part of your image analysis or download them. Just be sure to check out the rights and restrictions information about how you can distribute or display them.

The code below will acccess and save the images located the URLs you saved earlier. It saves them to a directory, so if you haven't already created a directory where you want to save the files, do that now. 

In [16]:
import os

This method works for situations where we you need a batch of images from one collection. However, if your images are from more than one collection, you might end up with files with the same name and end up overwriting them. And either way, these filenames don't tell you anything about the item. You might need to look up further metadata. So, here's an alternative approach that renames the file with the identifier used on the loc.gov website. We'll first re-fetch the image URL for each item and download the file, renaming it using the identifier. 

In [18]:
from urllib.parse import urlparse

def get_and_save_images(results_url, path):
    '''
    Takes as input the url for the collection or results set
    e.g. https://www.loc.gov/collections/baseball-cards
    and a list of items (used for pagination)

    Args:
        results_url (str): The url for the collection.
        path (str): The path in which images will be saved.
    '''
    #TODO: call API and store items in results variable
    params = {"fo": "json", "c": 100, "at": "results,pagination"} # remove the date parameter here to download the entire collection
    call = requests.get(results_url, params=params)
    data = call.json()
    results = data['results']
    
    #TODO: Find last image in image_url, and store in variable
    for result in results:
        # don't try to get images from the collection-level result or web page results
        if "collection" not in result.get("original_format") and "web page" not in result.get("original_format"):
            if result.get("image_url"):
                image = result.get("image_url")[-1]
                #TODO: create a filename with the identifier portion of the item URL, and save as path
                identifier = urlparse(result["id"])[2].rstrip('/')
                identifier = identifier.split('/')[-1]
                
                filename = f"{identifier}.jpg"
                filename = os.path.join(path, filename)
            
                #TODO: store image response and write to path
                image_response = requests.get(image, stream=True)
                with open(filename, 'wb') as fd:
                    for chunk in image_response.iter_content(chunk_size=100000):
                        fd.write(chunk)
    #TODO: Make sure we haven't hit the end of the pages
    if data["pagination"]["next"] is not None: # make sure we haven't hit the end of the pages
        next_url = data["pagination"]["next"]
        print(f"getting next page: {next_url}")
        time.sleep(3) # timer to avoid API rate limits
        get_and_save_images(next_url, path)

In [20]:
os.mkdir("images-named")

In [21]:
get_and_save_images("https://www.loc.gov/collections/baseball-cards/?dates=1888", "images-named")

### Connecting the image file to the metadata
The filename the code creates is the item's identifier, so you reconstruct a URL for the item's metadata. For example, to examine at the metadata for the first item in the list, 2007680728.jpg, you can add ``https://www.loc.gov/item/`` before the identifier. 

``https://www.loc.gov/item/2007680728``

You can also request the metadata in JSON format by adding ``?fo=json&at=item`` at the end. 

In [22]:
import pandas as pd
import requests

def save_metadata(results_url, path):

    params = {"fo": "json", "c": "100", "at": "results,pagination"} # remove the date parameter here to download the entire collection
    call = requests.get(results_url, params=params)
    data = call.json()
    results = data['results']

    df = pd.DataFrame()
    #TODO: Convert results to dataframe, use for loop to iterate and add to dataframe. Convert dataframe to CSV.
    for result in results:
        if "collection" not in result.get("original_format") and "web page" not in result.get("original_format"):
            df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)


    df.to_csv(path + '/metadata.csv', index=False)

    if data["pagination"]["next"] is not None: # make sure we haven't hit the end of the pages
        next_url = data["pagination"]["next"]
        print(f"getting next page: {next_url}")
        time.sleep(3) # timer to avoid API rate limits
        save_metadata(next_url, path)

In [23]:
import os
os.mkdir("metadata")

In [24]:
save_metadata("https://www.loc.gov/collections/baseball-cards/?dates=1888", "metadata")

## Conclusion

Access to batches of images opens up the door to new forms of analysis, research, and experimentation. 

We've shown where you can find information about rights and restrictions on use of images, how to find the image URLs in the API response, and how to save images if your analysis requires having the files on disk. 

Best of luck with your searching and let us know how your project goes!